# pandas 之一：Series 的创建

---

## 先说 pandas 是什么

Numpy 解决了"快速算数字"，但它有个短板：**没有标签**。
你拿到一个 `ndarray`，只能靠"第 0 个、第 1 个"这样的位置去猜这一列是什么意思。

pandas 就是在 Numpy 之上，给数据**贴上标签**：

| | Numpy | pandas |
|---|---|---|
| 一维 | `ndarray` | **`Series`**（带索引的一维数组） |
| 二维 | 二维 `ndarray` | **`DataFrame`**（带行列标签的表格） |
| 取值靠 | 位置下标 | **位置 + 标签** |

**一句话**：`Series = 一维数组 + 索引标签`。

你可以把 `Series` 想成"**带名字的一列 Excel 数据**"。

## 学习目标

1. 掌握 3 种创建 Series 的方式：列表、字典、已有 Series
2. 理解 `index`（索引）和 `name`（名称）这两个关键参数
3. 弄懂"用新索引套旧 Series"时为什么会出现 `NaN`
4. 知道 `Series` 和 `ndarray`、`list` 的区别


In [ ]:
import pandas as pd
import numpy as np

print('pandas 版本：', pd.__version__)


---
## 一、从列表创建（最基础）

不指定索引时，pandas 会自动生成 `0, 1, 2, ...` 的**位置索引**。


In [ ]:
s1 = pd.Series([1, 2, 3, 4, 5])
print(s1)


注意打印结果的格式：

```
0    1      <- 左边是【索引】，右边是【值】
1    2
2    3
...
dtype: int64   <- 底部标出整列的【数据类型】
```

**左边那一列不是数据，是索引。** 这是 Series 的灵魂。


In [ ]:
print('索引 =', list(s1.index))
print('值   =', list(s1.values))
print('类型 =', s1.dtype)


---
## 二、自定义索引（把位置换成有意义的标签）

这是 Series 相比 Numpy 数组最大的价值 —— **索引可以是任何东西**：
字符串、日期、ID……


In [ ]:
s2 = pd.Series([1, 2, 3, 4, 5], index=['a', 'b', 'c', 'd', 'e'])
print(s2)
print('\n索引：', list(s2.index))


现在取值可以"按名字"取了，不用记位置：


In [ ]:
print("s2['c'] =", s2['c'], '  <- 按标签取，比 s2[2] 直观得多')


### 索引不要求是唯一的


In [ ]:
s_dup = pd.Series([10, 20, 30], index=['x', 'x', 'y'])
print(s_dup)
print("\n取 s_dup['x'] 会返回【两个值】：")
print(s_dup['x'], '  <- 是 Series，不是单个数字')


### 索引长度必须和值一样长，否则报错


In [ ]:
try:
    bad = pd.Series([1, 2, 3], index=['a', 'b'])
except ValueError as e:
    print('报错了（这是对的）：')
    print(e)


---
## 三、`name` 参数：给这一列起个名字

`name` 看起来不起眼，但在**合并成 DataFrame**、**画图**、
**分组聚合**时，它会自动变成为列名 / 图例名，非常好用。


In [ ]:
s3 = pd.Series([1, 2, 3, 4, 5], name='月份')
print(s3)
print('\nname 属性 =', s3.name)


> **对比一下**：没有 name 和有 name 的打印结果，差别就在最后一行 ——
> 有 name 的会多出 `Name: 月份`。


In [ ]:
print(pd.Series([1, 2, 3]))
print()
print(pd.Series([1, 2, 3], name='月份'))


---
## 四、从字典创建（键自动变成索引）

这是**最常用**的创建方式 —— 因为字典本身就"有键有值"，天然对应"索引 + 数据"。


In [ ]:
s4 = pd.Series({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}, name='月份')
print(s4)


字典的 **键顺序在 Python 3.7+ 是有保证的**，
所以 Series 的索引顺序就是字典里写的顺序。


In [ ]:
s5 = pd.Series({'北京': 2154, '上海': 2487, '广州': 1530}, name='人口(万)')
print(s5)
print('\n键变成了索引：', list(s5.index))


---
## 五、用「新索引」套「旧 Series」—— 出现 NaN 的原理

这个机制**必须搞懂**，因为它是 pandas 里一切"对齐"行为的基础。

**规则：拿新索引去旧 Series 里逐个查，查到就取值，查不到就填 `NaN`。**


In [ ]:
s6 = pd.Series({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}, name='月份')

# 新索引里 'a' 存在，'B' 不存在（注意大小写！）
s7 = pd.Series(s6, index=['a', 'B'])
print(s7)


`'a'` 查到了 -> `1.0`；`'B'` 没查到 -> `NaN`。

**注意 dtype 变成了 `float64`** —— 因为 `NaN` 是浮点数，
整数列里一旦混入 `NaN`，整列就得升级成浮点才能装下。
这是 pandas 里最常见的一个"类型悄悄变了"的现象。


In [ ]:
print('dtype =', s7.dtype, '  <- 因为有了 NaN，从 int64 变成了 float64')


### ⚠️ 大小写敏感是个真实的坑

上面那个例子里 `'B'` 是**大写**，而字典里是**小写** `'b'`。
pandas **完全不做大小写转换**，它认为这是两个不同的标签。


In [ ]:
print("严格区分大小写：s6['b'] =", s6['b'], '，而 s6.get(\'B\') =', s6.get('B', '不存在'))


### 反过来：用旧 Series 的索引去筛

也可以只挑出部分索引，构造一个"子集 Series"：


In [ ]:
s8 = pd.Series(s6, index=['a', 'c', 'e'])
print('只保留 a、c、e：')
print(s8)


---
## 六、进阶：用标量创建一个"广播"的 Series


In [ ]:
s9 = pd.Series(100, index=['a', 'b', 'c'])
print('用一个标量 100 铺满 3 个索引：')
print(s9)


---
## 七、Series 和别的类型怎么互相转


In [ ]:
s10 = pd.Series([1, 2, 3, 4, 5], index=list('abcde'))

# Series -> 字典（最常用）
print('转字典：', s10.to_dict())

# Series -> 列表
print('转列表：', s10.tolist())

# Series -> numpy 数组
print('转 ndarray：', s10.to_numpy(), ' 类型', type(s10.to_numpy()))


> **实用场景**：要把 Series 当参数传给一个只认 dict 的函数时，
> `s.to_dict()` 一行就搞定，比手写循环清爽得多。


---
## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `pd.Series([1,2,3], index=['a','b'])` | `ValueError: Length of values...` | 索引长度必须等于值长度 |
| 用新索引套旧 Series 出现 `NaN` | 以为数据丢了 | 那是**没查到的标签**，属正常行为 |
| `'B'` 和 `'b'` 当成同一个 | 取不到值 / 得到 NaN | 索引**严格区分大小写** |
| 整数列加了 NaN 后 dtype 变了 | `int64 -> float64` | NaN 是浮点，会"传染"整列 |
| 以为 `name` 没用 | 合并/画图时列名变成 0 | 创建时就给 `name`，后面省事 |
| `s.values` 改了一个另一个也变 | 底层是同一块内存 | 要独立就用 `s.copy()` 或 `.to_numpy().copy()` |
| 用 `s[0]` 取自定义索引的 Series | 结果难以预测 | 自定义索引时**用标签取**，别用位置 |

---

## 九、动手练习

**练习 1**：创建一个 Series，值为 `[88, 92, 79, 95]`，索引为 `['小明','小红','小刚','小美']`，`name` 为 `'数学成绩'`，并打印出来。

**练习 2**：用字典创建一个"城市-人口"Series，然后：
1. 取出"上海"的人口
2. 转成字典
3. 打印它的索引和值

**练习 3**：给定 `s = pd.Series({'a':1,'b':2,'c':3})`，用索引 `['a','x','c']` 构造新 Series，
预测结果是什么，再运行验证。解释为什么 `'x'` 的位置是 `NaN`，以及 dtype 变成了什么。

**练习 4**：创建一个长度为 5 的 Series，全部值为 0，索引是 `['r1','r2','r3','r4','r5']`。

**练习 5**：创建一个 Series 包含重复索引 `['x','x','y']`，值 `[1,2,3]`，
取出 `'x'` 并说明返回的是什么类型。

---

**答案思路（先自己做）**

```python
# 1
s = pd.Series([88, 92, 79, 95],
              index=['小明', '小红', '小刚', '小美'],
              name='数学成绩')
print(s)

# 2
city = pd.Series({'北京': 2154, '上海': 2487})
print(city['上海'])
print(city.to_dict())
print(list(city.index), list(city.values))

# 3
s = pd.Series({'a': 1, 'b': 2, 'c': 3})
print(pd.Series(s, index=['a', 'x', 'c']))
# 'x' 没查到 -> NaN；因为出现 NaN，dtype 从 int64 变 float64

# 4
print(pd.Series(0, index=['r1', 'r2', 'r3', 'r4', 'r5']))

# 5
s = pd.Series([1, 2, 3], index=['x', 'x', 'y'])
print(s['x'])       # 返回 Series（两个值）
```
